In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
DETECTORS = ["AD11", "AD12", "AD21", "AD22", "AD31", "AD32", "AD33", "AD34"]

def get_events():
    base_path = "dayabay_full_dataset_npz_1-0-0/npz/dayabay_dataset"

    events = {}
    for detector in DETECTORS:
        data = np.load(f"{base_path}/dayabay_events_{detector}.npz")
        events[detector] = data["events"]
        data.close()

    return events

In [3]:
events = get_events()

In [4]:
# из вектора получить единичный
unit = lambda x: x/np.linalg.norm(x)

Из статьи найдём радиус-вектора ко всем детекторам/реакторам. Вычтя из одного другое найдём радиус-вектор от детектора к реактору (в координатах которые в статье даны). Если надо, как-то пересчитаем это в цилиндрические координаты каждого детектора(как?)

Направление протекания IBD будем считать как разницу радиус-вектора от delayed- к prompt-сигналу.

Для каждого детектора найдём "среднее" направление протекания IBD (взвешенное (т.е., получив массив векторов dr от delayed- к prompt-сигналу, сначала найти среднее, а потом сделать из этого единичный вектор) и невзвешенное (для каждого dr сначала посчитать единичный вектор от delayed- к prompt-сигналу, и только затем усреднить))

Это направление (если ненулевое) сопоставим с радиус-вектором к ближайшему реактору (влиянием остальных пренебрежем, наверное)

In [5]:
def get_detector_plot(detector):
    r_p = events[detector][["vertex_prompt_x_mm", "vertex_prompt_y_mm", "vertex_prompt_z_mm"]]
    x_p = r_p["vertex_prompt_x_mm"]
    y_p = r_p["vertex_prompt_y_mm"]
    z_p = r_p["vertex_prompt_z_mm"]


    r_d = events[detector][["vertex_delayed_x_mm", "vertex_delayed_y_mm", "vertex_delayed_z_mm"]]
    x_d = r_d["vertex_delayed_x_mm"]
    y_d = r_d["vertex_delayed_y_mm"]
    z_d = r_d["vertex_delayed_z_mm"]


    dx = x_d - x_p
    dy = y_d - y_p
    dz = z_d - z_p
    l = np.sqrt(dx**2 + dy**2 + dz**2)

    # радиус вектор (предполагаемого) направления протекания IBD
    dr = np.vstack((dx, dy, dz)).T
    # единичный вектор направления IBD
    u = np.vstack((dx/l, dy/l, dz/l)).T

    # polar
    theta = np.acos(dz/l)
    # azimuthal angle for each dr
    phi = np.arctan2(dy, dx)
    
    # азимутальный угол усредненного единичного вектора 
    phi1 = np.arctan2(u.mean(axis=0)[1], u.mean(axis=0)[0])

    print(f"\n{'=' * 45}")
    print(f" Statistics for {detector}")
    print(f"{'=' * 45}")

    print(f"  Unit vector of mean dR:        {unit(dr.mean(axis=0))}")
    print(f"  Mean of unit vectors:          {unit(u.mean(axis=0))}")

    print(f"  Average polar angle:           {np.degrees(theta.mean()):8.2f}°")
    print(f"  Average azimuthal angle (IBD): {np.degrees(phi.mean()):8.2f}°")
    print(f"  Azimuth of mean unit vector:   {np.degrees(phi1) % 360:8.2f}°")

    print(f"{'=' * 45}\n")

    # plt.show()

In [6]:
# радиус-вектора к реакторам/детекторам взяты из таблицы в статье
reactors = np.load("dayabay_reactors.npz")
detectors = np.load("dayabay_detectors.npz")
R = reactors["reactor_xyz_m"]
D = detectors["detector_xyz_m"]


for i, detector in enumerate(DETECTORS):
    get_detector_plot(detector)

    # d = D[i] - R[i]
    # радиус-вектор от детектора к реактору, но в другой координатной системе, это надо пересчитать (как-то)




 Statistics for AD11
  Unit vector of mean dR:        [ 0.7877911  -0.52451205  0.32291237]
  Mean of unit vectors:          [ 0.81437236 -0.5788112   0.04213403]
  Average polar angle:              89.91°
  Average azimuthal angle (IBD):    -2.35°
  Azimuth of mean unit vector:     324.60°


 Statistics for AD12
  Unit vector of mean dR:        [ 0.74661505 -0.60844696  0.268995  ]
  Mean of unit vectors:          [ 0.80070066 -0.598425    0.0276769 ]
  Average polar angle:              89.93°
  Average azimuthal angle (IBD):    -2.63°
  Azimuth of mean unit vector:     323.23°


 Statistics for AD21
  Unit vector of mean dR:        [-0.70389485 -0.68383366  0.1921028 ]
  Mean of unit vectors:          [-0.7376261  -0.67454803  0.02987854]
  Average polar angle:              89.92°
  Average azimuthal angle (IBD):    -3.58°
  Azimuth of mean unit vector:     222.44°


 Statistics for AD22
  Unit vector of mean dR:        [-0.754733   -0.6311215   0.17906313]
  Mean of unit vectors:  

Как видим, два усреднения дали слишком разные результаты

1.Для обоих посчитанных углов видно: для близких детекторов углы примерно равны, а для далёких они различаются

2.Углы ненулевые, т.е. у IBD есть направленность. Причем из (1) она, предположительно, зависит от географического положения детектора относительно реакторов